# Yuda 7 — Stacking Ensemble
Learned ensemble (LogisticRegression) vs weighted average champion

In [1]:
import sys
sys.path.insert(0, "../..")

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import json
import gc
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from torch.utils.data import Dataset, DataLoader

import open_clip
import warnings
warnings.filterwarnings('ignore')

from modules.utils.config import *
from modules.utils.seed import set_seed
from modules.utils.load_data import load_train
from modules.models.factory import TrashClassifier
from modules.data.dataset import get_dataloaders, get_transforms

set_seed(SEED)
print(f"Device: {DEVICE}")
print(f"open_clip: {open_clip.__version__}")

/home/prayudahlah/projects/external/big-data-big-trouble/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
open_clip: 3.3.0


In [2]:
_CLASS_TO_IDX = {'0_Recyclable': 0, '1_Electronic': 1, '2_Organic': 2}
CLASS_NAMES = ['Recyclable', 'Electronic', 'Organic']

CLIP_NAME = 'ViT-B-32'
CLIP_PRETRAINED = 'laion2b_s34b_b79k'
clip_model, _, clip_transform = open_clip.create_model_and_transforms(
    CLIP_NAME, pretrained=CLIP_PRETRAINED
)
clip_tokenizer = open_clip.get_tokenizer(CLIP_NAME)
clip_model = clip_model.to('cpu')
clip_model.eval()
clip_visual = clip_model.visual.to(DEVICE)
clip_dim = clip_visual.output_dim
print(f"CLIP dim: {clip_dim}")

prompts_improved = [
    "a photo of recyclable items like paper, plastic, glass, and metal",
    "a photo of electronic waste like computers, phones, cables, and batteries",
    "a photo of organic waste like food scraps, leaves, and plants",
]

class CLIPWithTextFeatures(nn.Module):
    def __init__(self, clip_model, clip_visual, prompts, num_classes=3):
        super().__init__()
        self.visual = clip_visual
        self.encoder = self.visual
        self.logit_scale = clip_model.logit_scale
        dim = self.visual.output_dim
        text_tokens = clip_tokenizer(prompts)
        with torch.no_grad():
            text_feat = clip_model.encode_text(text_tokens)
            text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
        self.register_buffer('text_features', text_feat)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(dim + 3, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        v = self.visual(x)
        v_norm = v / v.norm(dim=-1, keepdim=True)
        sim = v_norm @ self.text_features.T
        combined = torch.cat([v, sim * self.logit_scale.exp()], dim=1)
        return self.classifier(combined)
    def freeze_encoder(self):
        for p in self.visual.parameters():
            p.requires_grad = False
    def unfreeze_encoder(self):
        for p in self.visual.parameters():
            p.requires_grad = True


def get_probs(model, loader):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for x, _ in tqdm(loader, desc="Inference", leave=False):
            x = x.to(DEVICE)
            logits = model(x)
            all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(all_probs)

conv_val_tfm = get_transforms(augment=False, img_size=224)
conv_val_tfm_300 = get_transforms(augment=False, img_size=300)

CLIP dim: 512


---
## Dataloaders

In [3]:
train_loader, val_loader, val_ds = get_dataloaders(batch_size=32, num_workers=4)
df_train = train_loader.dataset.df
df_val = val_loader.dataset.df
print(f"Train: {len(df_train)} | Val: {len(df_val)}")

class SingleTransformDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(PROJECT_ROOT / row['path']).convert('RGB')
        return self.transform(img), _CLASS_TO_IDX[row['label']]

clip_val_ds = SingleTransformDataset(df_val, clip_transform)
clip_val_loader = DataLoader(clip_val_ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

conv_val_ds = SingleTransformDataset(df_val, conv_val_tfm)
conv_val_loader = DataLoader(conv_val_ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

conv_val_ds_300 = SingleTransformDataset(df_val, conv_val_tfm_300)
conv_val_loader_300 = DataLoader(conv_val_ds_300, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)

val_labels = np.array([_CLASS_TO_IDX[r['label']] for _, r in df_val.iterrows()])
print(f"Val labels: {len(val_labels)}")

Train: 21221 | Val: 5306
Val labels: 5306


---
## Load 3 Champion Models + Collect Val Probs

In [4]:
models_to_load = [
    ('C_focal', 'yuda6_prompt_focal', 'clip', None),
    ('ConvNeXt', 'yuda_convnext_tiny', 'conv', 224),
    ('EffNet', 'yuda2_effnet_b3_300', 'conv', 300),
]

all_probs = {}
print("Loading models + generating val probs...")
for name, ckpt_name, model_type, img_size in models_to_load:
    if model_type == 'clip':
        model = CLIPWithTextFeatures(clip_model, clip_visual, prompts_improved).to(DEVICE)
        loader = clip_val_loader
    else:
        arch = 'convnext_tiny' if 'ConvNeXt' in name else 'efficientnet_b3'
        model = TrashClassifier(arch, num_classes=3).to(DEVICE)
        loader = conv_val_loader_300 if img_size == 300 else conv_val_loader
    
    ckpt = torch.load(RESULTS / f'{ckpt_name}.pt', map_location=DEVICE, weights_only=True)
    model.load_state_dict(ckpt)
    model.eval()
    
    with open(RESULTS / f'{ckpt_name}.json') as f:
        meta = json.load(f)
    print(f"  {name}: loaded (val_f1={meta.get('best_val_f1', '?'):.4f})")
    
    all_probs[name] = get_probs(model, loader)
    del model
    torch.cuda.empty_cache()
    gc.collect()

for name, probs in all_probs.items():
    print(f"  {name}: {probs.shape}")

Loading models + generating val probs...
  C_focal: loaded (val_f1=0.9839)


  ConvNeXt: loaded (val_f1=0.9742)


  EffNet: loaded (val_f1=0.9731)


  C_focal: (5306, 3)
  ConvNeXt: (5306, 3)
  EffNet: (5306, 3)


---
## Baseline: Reproduce Champion (Weighted Average Grid Search)

In [5]:
print("=" * 60)
print("Weighted Average Grid Search — Reproduce Champion")
print("=" * 60)

c_probs = all_probs['C_focal']
conv_probs = all_probs['ConvNeXt']
eff_probs = all_probs['EffNet']

rows = []

# 2-model: C + ConvNeXt
for w_c in np.arange(0.50, 0.91, 0.05):
    w_conv = 1.0 - w_c
    ensemble = w_c * c_probs + w_conv * conv_probs
    f1 = f1_score(val_labels, ensemble.argmax(axis=1), average='macro')
    rows.append({'name': f'C{w_c:.2f}+Conv{w_conv:.2f}', 'best_val_f1': f1})

# 3-model: C + ConvNeXt + EffNet (champion config)
for w_c in [0.5, 0.6, 0.7, 0.8]:
    for w_conv in [0.1, 0.2, 0.3]:
        w_eff = 1.0 - w_c - w_conv
        if w_eff < 0.05:
            continue
        ensemble = w_c * c_probs + w_conv * conv_probs + w_eff * eff_probs
        f1 = f1_score(val_labels, ensemble.argmax(axis=1), average='macro')
        rows.append({'name': f'C{w_c:.2f}+Conv{w_conv:.2f}+Eff{w_eff:.2f}', 'best_val_f1': f1})

# Also include single models
for name, probs in all_probs.items():
    f1 = f1_score(val_labels, probs.argmax(axis=1), average='macro')
    rows.append({'name': name, 'best_val_f1': f1})

df_wavg = pd.DataFrame(rows).sort_values('best_val_f1', ascending=False).reset_index(drop=True)
# rename old champion entries to match yuda_6 naming
print("\nTop 5:")
print(df_wavg.head(10).to_string(index=False))

champion = df_wavg.iloc[0]
print(f"\n{'=' * 60}")
print(f"🏆 Champion reproduced: {champion['name']} @ {champion['best_val_f1']:.4f}")
print(f"   Expected (yuda_6): C0.70+Conv0.10+Eff0.20 @ 0.9856")

Weighted Average Grid Search — Reproduce Champion

Top 5:
                  name  best_val_f1
C0.70+Conv0.10+Eff0.20     0.985616
C0.80+Conv0.10+Eff0.10     0.984873
C0.60+Conv0.10+Eff0.30     0.984695
        C0.80+Conv0.20     0.984420
C0.60+Conv0.20+Eff0.20     0.984398
        C0.75+Conv0.25     0.984273
        C0.85+Conv0.15     0.983971
               C_focal     0.983950
C0.70+Conv0.20+Eff0.10     0.983676
C0.60+Conv0.30+Eff0.10     0.983506

🏆 Champion reproduced: C0.70+Conv0.10+Eff0.20 @ 0.9856
   Expected (yuda_6): C0.70+Conv0.10+Eff0.20 @ 0.9856


---
## Stacking: Logistic Regression on 9 Probabilities

In [8]:
print("=" * 60)
print("Stacking — Logistic Regression (9 features)")
print("=" * 60)

# Features: concat probs from 3 models: shape (5306, 9)
X = np.concatenate([c_probs, conv_probs, eff_probs], axis=1)
y = val_labels

print(f"Feature matrix: {X.shape} (val_samples, 3 models × 3 classes)")

# Train meta-classifier
meta = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=42)
meta.fit(X, y)
preds_stack = meta.predict(X)

f1_stack = f1_score(y, preds_stack, average='macro')
f1_stack_per = f1_score(y, preds_stack, average=None)
print(f"\nStacking F1: {f1_stack:.4f}")
print(f"Per-class F1: Recyclable={f1_stack_per[0]:.4f}, Electronic={f1_stack_per[1]:.4f}, Organic={f1_stack_per[2]:.4f}")

print("\nConfusion Matrix:")
cm = confusion_matrix(y, preds_stack)
print(cm)
print("\nNormalized (row):")
print((cm.astype(float) / cm.sum(axis=1, keepdims=True)).round(4))

Stacking — Logistic Regression (9 features)
Feature matrix: (5306, 9) (val_samples, 3 models × 3 classes)

Stacking F1: 0.9860
Per-class F1: Recyclable=0.9790, Electronic=0.9949, Organic=0.9841

Confusion Matrix:
[[1960    1   39]
 [   5  786    1]
 [  39    1 2474]]

Normalized (row):
[[9.800e-01 5.000e-04 1.950e-02]
 [6.300e-03 9.924e-01 1.300e-03]
 [1.550e-02 4.000e-04 9.841e-01]]


---
## Learned Coefficients — Per-class Model Contribution

In [9]:
print("=" * 60)
print("Coefficients Analysis")
print("=" * 60)

model_names = ['C_focal', 'ConvNeXt', 'EffNet']
class_names = ['Recyclable (0)', 'Electronic (1)', 'Organic (2)']
coef = meta.coef_  # (3, 9)
intercept = meta.intercept_

print(f"\nCoefficient matrix ({len(class_names)} classes × {len(model_names)*3} features):")
for ci, cname in enumerate(class_names):
    print(f"\n  {cname}:")
    total_contrib = 0
    for mi, mname in enumerate(model_names):
        w = coef[ci, mi*3:(mi+1)*3]
        print(f"    {mname:10s}: p0={w[0]:+.4f}  p1={w[1]:+.4f}  p2={w[2]:+.4f}  |sum|={np.abs(w).sum():.4f}")
        total_contrib += np.abs(w).sum()
    print(f"    {'':10s}  intercept={intercept[ci]:+.4f}  total_contrib={total_contrib:.4f}")

print(f"\n{'=' * 60}")
print(f"Stacking F1: {f1_stack:.4f}")
print(f"Best weighted avg F1: {champion['best_val_f1']:.4f}")
print(f"Delta: {f1_stack - champion['best_val_f1']:+.4f}")

Coefficients Analysis

Coefficient matrix (3 classes × 9 features):

  Recyclable (0):
    C_focal   : p0=+3.0990  p1=-1.3290  p2=-1.7050  |sum|=6.1329
    ConvNeXt  : p0=+0.6277  p1=-0.5616  p2=-0.0010  |sum|=1.1903
    EffNet    : p0=+0.6553  p1=-0.6237  p2=+0.0334  |sum|=1.3123
                intercept=+0.1439  total_contrib=8.6356

  Electronic (1):
    C_focal   : p0=-1.2576  p1=+2.1896  p2=-1.0964  |sum|=4.5436
    ConvNeXt  : p0=-0.8365  p1=+0.9725  p2=-0.3004  |sum|=2.1094
    EffNet    : p0=-0.5026  p1=+1.6620  p2=-1.3238  |sum|=3.4883
                intercept=-0.3549  total_contrib=10.1413

  Organic (2):
    C_focal   : p0=-1.8414  p1=-0.8607  p2=+2.8014  |sum|=5.5034
    ConvNeXt  : p0=+0.2088  p1=-0.4109  p2=+0.3014  |sum|=0.9211
    EffNet    : p0=-0.1527  p1=-1.0383  p2=+1.2904  |sum|=2.4814
                intercept=+0.2110  total_contrib=8.9059

Stacking F1: 0.9860
Best weighted avg F1: 0.9856
Delta: +0.0004


---
## Summary

In [10]:
print("=" * 60)
print("Yuda 7 — Final Summary")
print("=" * 60)

all_rows = []
for _, r in df_wavg.iterrows():
    all_rows.append({'name': r['name'], 'best_val_f1': r['best_val_f1'], 'method': 'WeightedAvg'})
all_rows.append({'name': 'Stacking_LR', 'best_val_f1': f1_stack, 'method': 'Stacking'})

summary = pd.DataFrame(all_rows).sort_values('best_val_f1', ascending=False).reset_index(drop=True)
summary.to_csv(RESULTS / 'yuda7_summary.csv', index=False)

print("\nTop 5:")
print(summary.head(10).to_string(index=False))

winner = summary.iloc[0]
print(f"\n{'=' * 60}")
print(f"🏆 Winner: {winner['name']} ({winner['method']}) @ {winner['best_val_f1']:.4f}")

# Compare
method_wavg = summary[summary['method'] == 'WeightedAvg']['best_val_f1'].max()
method_stack = summary[summary['method'] == 'Stacking']['best_val_f1'].max()
print(f"\nComparison:")
print(f"  Best WeightedAvg:     {method_wavg:.4f}")
print(f"  Best Stacking (LR):  {method_stack:.4f}")
print(f"  Gain:                +{method_stack - method_wavg:.4f}")
print(f"  yuda_6 champion:     0.9856")
print(f"  Improvement:         +{winner['best_val_f1'] - 0.9856:.4f}")

Yuda 7 — Final Summary

Top 5:
                  name  best_val_f1      method
           Stacking_LR     0.986016    Stacking
C0.70+Conv0.10+Eff0.20     0.985616 WeightedAvg
C0.80+Conv0.10+Eff0.10     0.984873 WeightedAvg
C0.60+Conv0.10+Eff0.30     0.984695 WeightedAvg
        C0.80+Conv0.20     0.984420 WeightedAvg
C0.60+Conv0.20+Eff0.20     0.984398 WeightedAvg
        C0.75+Conv0.25     0.984273 WeightedAvg
        C0.85+Conv0.15     0.983971 WeightedAvg
               C_focal     0.983950 WeightedAvg
C0.70+Conv0.20+Eff0.10     0.983676 WeightedAvg

🏆 Winner: Stacking_LR (Stacking) @ 0.9860

Comparison:
  Best WeightedAvg:     0.9856
  Best Stacking (LR):  0.9860
  Gain:                +0.0004
  yuda_6 champion:     0.9856
  Improvement:         +0.0004
